# gru_va board session #2 — aliasing battery + probes + 15 marginal pairs (REV1)
Sentinel: BOARDS2-REV1

Built on gru_campaign_102_REV1 certified semantics: Overlay+clock-fix as one
inseparable action, DMA order arm-recv -> start -> send, chunked state carry
with reset_state=1 on chunk 0 only, fsync'd resumable results CSV.

Job sources:
1. **Aliasing battery**: campaign_manifest.csv filtered to STIM_CELL (default
   rodent_max), every width x 5 stimulus files (tones/sweep/silence/impulse/edge).
2. **Marginal 15**: marginal15.csv (cell,width,bit,ref) — golden gate ONLY.
No throughput reps in this session (latency already fleet-characterized).

## 1. Configuration — edit here, nowhere else

In [ ]:
import csv, hashlib, os, time
import numpy as np
from pynq import Overlay, allocate
from pynq.ps import Clocks

SESSION_DIR = "/home/xilinx/jupyter_notebooks/session2"
MANIFEST    = os.path.join(SESSION_DIR, "campaign_manifest.csv")   # reused from campaign102
MARGINAL15  = os.path.join(SESSION_DIR, "marginal15.csv")          # cell,width,bit,ref (golden-gate only)
RESULTS     = os.path.join(SESSION_DIR, "session2_results.csv")
GOLDEN_IN   = os.path.join(SESSION_DIR, "golden_input.bin")        # 4096 f32, shared stimulus
IP_NAME     = "gru_va_0"
DMA_NAME    = "axi_dma_0"
CHUNK       = 65536
FCLK_MHZ    = 100.0
GOLDEN_N    = 4096
SR          = 48000.0
STIM_CELL   = "rodent_max"      # aliasing battery cell

# stimulus registry: short name -> filename (STIMGEN-REV1 outputs)
STIMS = {
    "tones":   "stim_tones_in.f32",
    "sweep":   "stim_sweep_in.f32",
    "silence": "stim_silence_in.f32",
    "impulse": "stim_impulse_in.f32",
    "edge":    "stim_edge_in.f32",
}

st = os.statvfs(SESSION_DIR)
print("free space: %.1f GB" % (st.f_bavail*st.f_frsize/1e9))
print("outputs per width: ~27 MB x all stims; 17 widths ~ 0.5 GB total")

## 2. Helpers (run once per kernel session)

In [ ]:
def md5_of(arr):
    return hashlib.md5(arr.tobytes()).hexdigest().upper()

def start_kernel(ip):
    ip.register_map.CTRL.AP_START = 1

def wait_done(ip, timeout_s=15.0):
    t0 = time.time()
    while int(ip.register_map.CTRL.AP_IDLE) != 1:
        if time.time() - t0 > timeout_s:
            raise TimeoutError("kernel not idle after %.1fs" % timeout_s)

def load_bitstream(bit_name):
    """Overlay load + clock fix + handle fetch, as one inseparable action."""
    ol = Overlay(os.path.join(SESSION_DIR, bit_name))
    Clocks.fclk0_mhz = FCLK_MHZ
    assert abs(Clocks.fclk0_mhz - FCLK_MHZ) < 0.5, "fclk0 set failed: %s" % Clocks.fclk0_mhz
    ip  = getattr(ol, IP_NAME)
    dma = getattr(ol, DMA_NAME)
    return ol, ip, dma

def run_golden(ip, dma, golden_x, ref=None):
    """4096-sample gate. Returns (board_out, row_fragment dict)."""
    frag = {}
    ibuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    obuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    try:
        ibuf[:] = golden_x; ibuf.flush()
        rm = ip.register_map
        rm.mode = 1; rm.n_samples = GOLDEN_N; rm.reset_state = 1
        dma.recvchannel.transfer(obuf); start_kernel(ip); dma.sendchannel.transfer(ibuf)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obuf.invalidate()
        bg = np.asarray(obuf).copy()
    finally:
        ibuf.freebuffer(); obuf.freebuffer()
    frag["golden_md5"] = md5_of(bg)
    if ref is not None:
        d = np.abs(bg - ref)
        frag["max_abs_err"] = "%.6e" % float(d.max())
        frag["worst_idx"]   = str(int(d.argmax()))
        frag["gate"] = "PASS" if np.array_equal(bg, ref) else "FAIL"
    else:
        frag["gate"] = "NO_REF"
    return bg, frag

def run_stim_file(ip, dma, ibc, obc, x, out_buf):
    """Chunked state-carried run over one stimulus; reset on chunk 0 only.
    Returns row_fragment dict. Identical transport semantics to the certified
    campaign full-length loop."""
    rm = ip.register_map
    n_total = x.size
    t0 = time.time(); pos = 0; ci = 0
    while pos < n_total:
        n = min(CHUNK, n_total - pos)
        ibc[:n] = x[pos:pos+n]; ibc.flush()
        rm.mode = 1; rm.n_samples = n; rm.reset_state = 1 if ci == 0 else 0
        dma.recvchannel.transfer(obc[:n] if n < CHUNK else obc)
        start_kernel(ip)
        dma.sendchannel.transfer(ibc[:n] if n < CHUNK else ibc)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obc.invalidate()
        out_buf[pos:pos+n] = obc[:n]
        pos += n; ci += 1
    dt = time.time() - t0
    return {"n_samples": str(n_total), "secs": "%.2f" % dt,
            "ksamp_s": "%.0f" % (n_total/dt/1e3),
            "out_md5": md5_of(out_buf[:n_total])}

FIELDS = ["timestamp","cell","width","bit","stim","status","gate",
          "max_abs_err","worst_idx","golden_md5","out_md5","out_file",
          "n_samples","secs","ksamp_s","fclk_mhz","note"]

def append_row(row):
    new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row); f.flush(); os.fsync(f.fileno())

def load_done_set():
    done = set()
    if os.path.exists(RESULTS):
        with open(RESULTS) as f:
            for r in csv.DictReader(f):
                if r.get("status") == "OK":
                    done.add((r["cell"], int(r["width"]), r["stim"]))
    return done

golden_x = np.fromfile(GOLDEN_IN, dtype=np.float32)
assert golden_x.size == GOLDEN_N, "golden_input.bin wrong size: %d" % golden_x.size

stim_data = {}
for sname, fn in STIMS.items():
    p = os.path.join(SESSION_DIR, fn)
    assert os.path.exists(p), "MISSING stimulus: %s" % p
    stim_data[sname] = np.fromfile(p, dtype=np.float32)
    print("stim %-8s %9d samples  %s" % (sname, stim_data[sname].size, fn))
max_stim = max(a.size for a in stim_data.values())
out_buf = np.empty(max_stim, dtype=np.float32)
print("helpers defined; golden %d samples" % golden_x.size)

---

In [ ]:
# J1. Build job list + show resume status (safe to run any time)
jobs = []          # aliasing battery: full stim set
gate_only = []     # marginal 15: golden gate only

with open(MANIFEST) as f:
    for r in csv.DictReader(f):
        if r["cell"] == STIM_CELL:
            jobs.append({"cell": r["cell"], "width": int(r["width"]),
                         "bit": r["bit"], "ref": r.get("ref", "")})
jobs.sort(key=lambda e: e["width"])

if os.path.exists(MARGINAL15):
    with open(MARGINAL15) as f:
        for r in csv.DictReader(f):
            gate_only.append({"cell": r["cell"], "width": int(r["width"]),
                              "bit": r["bit"], "ref": r.get("ref", "")})
    print("marginal15: %d gate-only pairs" % len(gate_only))
else:
    print("marginal15.csv not present -- gate-only lane skipped")

done = load_done_set()
n_stim_todo = sum(1 for e in jobs for s in STIMS if (e["cell"], e["width"], s) not in done)
n_gate_todo = sum(1 for e in gate_only if (e["cell"], e["width"], "golden") not in done)
print("aliasing battery: %d bitstreams x %d stims -> %d runs todo" % (len(jobs), len(STIMS), n_stim_todo))
print("gate-only lane  : %d todo" % n_gate_todo)
missing = [e["bit"] for e in jobs + gate_only
           if not os.path.exists(os.path.join(SESSION_DIR, e["bit"]))]
if missing:
    print("MISSING BITSTREAMS (stage .bit AND matching .hwh before running):")
    for b in missing: print("   ", b)
else:
    print("all bitstreams present")

In [ ]:
# J2. THE SESSION LOOP -- start it and leave it alone
done = load_done_set()

# lane 1: aliasing battery (golden gate per load, then every pending stim)
for i, e in enumerate(jobs):
    pend = [s for s in STIMS if (e["cell"], e["width"], s) not in done]
    tag = "%s_w%d" % (e["cell"], e["width"])
    if not pend and (e["cell"], e["width"], "golden") in done:
        print("[%s] %d/%d %-22s all done, skip" % (time.strftime("%H:%M"), i+1, len(jobs), tag))
        continue
    ibc = obc = None
    try:
        ol, ip, dma = load_bitstream(e["bit"])
        ref = None
        if e["ref"]:
            rp = os.path.join(SESSION_DIR, e["ref"])
            if os.path.exists(rp) and os.path.getsize(rp) == 4*GOLDEN_N:
                ref = np.fromfile(rp, dtype=np.float32)
        bg, gfrag = run_golden(ip, dma, golden_x, ref)
        bg.tofile(os.path.join(SESSION_DIR, "s2_gold_%s.f32" % tag))
        if (e["cell"], e["width"], "golden") not in done:
            row = {k: "" for k in FIELDS}
            row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                       width=e["width"], bit=e["bit"], stim="golden", status="OK",
                       fclk_mhz="%.1f" % Clocks.fclk0_mhz, **gfrag)
            append_row(row)
        if gfrag["gate"] == "FAIL":
            print("[%s] %-22s GOLDEN GATE FAIL -- stims skipped" % (time.strftime("%H:%M"), tag))
            continue
        ibc = allocate(shape=(CHUNK,), dtype=np.float32)
        obc = allocate(shape=(CHUNK,), dtype=np.float32)
        for sname in pend:
            row = {k: "" for k in FIELDS}
            row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                       width=e["width"], bit=e["bit"], stim=sname,
                       gate=gfrag["gate"], golden_md5=gfrag["golden_md5"],
                       fclk_mhz="%.1f" % Clocks.fclk0_mhz)
            try:
                frag = run_stim_file(ip, dma, ibc, obc, stim_data[sname], out_buf)
                oname = "s2_%s_%s.f32" % (sname, tag)
                out_buf[:stim_data[sname].size].tofile(os.path.join(SESSION_DIR, oname))
                row.update(frag); row["out_file"] = oname; row["status"] = "OK"
            except TimeoutError as ex:
                row.update(status="TIMEOUT", note=str(ex))
            except Exception as ex:
                row.update(status="ERROR", note=type(ex).__name__ + ": " + str(ex)[:120])
            append_row(row)
            print("[%s] %d/%d %-22s %-8s %-7s %s k/s" % (time.strftime("%H:%M"),
                  i+1, len(jobs), tag, sname, row["status"], row.get("ksamp_s","-")))
    except Exception as ex:
        row = {k: "" for k in FIELDS}
        row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                   width=e["width"], bit=e["bit"], stim="load", status="ERROR",
                   note=type(ex).__name__ + ": " + str(ex)[:120])
        append_row(row)
        print("[%s] %-22s LOAD ERROR: %s" % (time.strftime("%H:%M"), tag, ex))
    finally:
        for b in (ibc, obc):
            try:
                if b is not None: b.freebuffer()
            except Exception:
                pass

# lane 2: marginal 15 -- golden gate only
for i, e in enumerate(gate_only):
    if (e["cell"], e["width"], "golden") in done:
        continue
    tag = "%s_w%d" % (e["cell"], e["width"])
    row = {k: "" for k in FIELDS}
    row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
               width=e["width"], bit=e["bit"], stim="golden", note="marginal15")
    try:
        ol, ip, dma = load_bitstream(e["bit"])
        ref = None
        if e["ref"]:
            rp = os.path.join(SESSION_DIR, e["ref"])
            if os.path.exists(rp) and os.path.getsize(rp) == 4*GOLDEN_N:
                ref = np.fromfile(rp, dtype=np.float32)
        bg, gfrag = run_golden(ip, dma, golden_x, ref)
        bg.tofile(os.path.join(SESSION_DIR, "s2_gold_%s.f32" % tag))
        row.update(gfrag); row["fclk_mhz"] = "%.1f" % Clocks.fclk0_mhz
        row["status"] = "OK"
    except TimeoutError as ex:
        row.update(status="TIMEOUT", note="marginal15; " + str(ex))
    except Exception as ex:
        row.update(status="ERROR", note="marginal15; " + type(ex).__name__ + ": " + str(ex)[:120])
    append_row(row)
    print("[%s] m15 %d/%d %-22s %-7s gate=%s" % (time.strftime("%H:%M"),
          i+1, len(gate_only), tag, row["status"], row.get("gate","-")))

print("session #2 pass complete -- results in", RESULTS)

In [ ]:
# J3. Results summary
import collections
if os.path.exists(RESULTS):
    rows = list(csv.DictReader(open(RESULTS)))
    byst = collections.Counter((r["status"], r["stim"]=="golden") for r in rows)
    print("rows:", len(rows))
    ok = [r for r in rows if r["status"]=="OK"]
    gates = collections.Counter(r["gate"] for r in ok if r["stim"]=="golden")
    print("golden gates:", dict(gates))
    st = collections.Counter(r["stim"] for r in ok if r["stim"]!="golden")
    print("stim runs OK:", dict(st))
    bad = [r for r in rows if r["status"]!="OK"]
    for r in bad: print("  BAD:", r["cell"], r["width"], r["stim"], r["status"], r["note"])
else:
    print("no results yet")

## Cleanup
Pull everything with verified_move.py (VMOVE-REV2) from Lenny; board-side
md5sum is ground truth. Outputs: s2_gold_*.f32 (16 KB each) and
s2_<stim>_<cell>_w<W>.f32 (0.6-18.6 MB each). Do not hand-delete; use the
emitted remove_verified.sh after transfer verification.